# Saba Yisroel Whisper Fine-Tuning
Fine-tune Whisper on 55 min of high-confidence Saba audio.

NNNNM!

In [ ]:
!pip install -q transformers datasets accelerate evaluate jiwer soundfile librosa
print("Done!")

In [ ]:
# Step 2: Download training data from ajew.org
import urllib.request, json
for f in ["saba_training_pairs.json","saba_lexicon.json","saba_vocabulary.txt"]:
    urllib.request.urlretrieve(f"https://ajew.org/data/saba-training/{f}", f)
    print(f"Downloaded {f}")
with open("saba_training_pairs.json") as fh:
    pairs = json.load(fh)
print(f"Training segments: {len(pairs)}")

In [ ]:
# Step 3: Mount Google Drive
from google.colab import drive
drive.mount("/content/drive")
import os
# The shared folder should be accessible
for root,dirs,files in os.walk("/content/drive/MyDrive"):
    mp3s=[f for f in files if f.endswith(".mp3")]
    if len(mp3s)>5:
        RECORDINGS=root
        print(f"Found {len(mp3s)} recordings in {root}")
        break

In [ ]:
dataset = Dataset.from_dict({"audio": [s["audio"] for s in segments], "text": [s["text"] for s in segments]})
split = dataset.train_test_split(test_size=0.1, seed=42)
print(f"Train: {len(split[chr(116)+chr(114)+chr(97)+chr(105)+chr(110)])}, Eval: {len(split[chr(116)+chr(101)+chr(115)+chr(116)])}")

In [ ]:
processor = WhisperProcessor.from_pretrained("openai/whisper-large-v3")
model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-large-v3")
model.config.forced_decoder_ids = processor.get_decoder_prompt_ids(language="he", task="transcribe")
print("Model loaded!")

In [ ]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer
import evaluate, torch

training_args = Seq2SeqTrainingArguments(output_dir="./saba-whisper", per_device_train_batch_size=4, gradient_accumulation_steps=2, learning_rate=1e-5, warmup_steps=50, max_steps=500, fp16=True, evaluation_strategy="steps", eval_steps=100, save_steps=100, logging_steps=25, predict_with_generate=True, generation_max_length=225, load_best_model_at_end=True, metric_for_best_model="wer", greater_is_better=False)
print("Config ready!")

In [ ]:
print("Training Saba-tuned Whisper...")
print("NNNNM!")
trainer.train()
print("Done!")

In [ ]:
trainer.save_model("./saba-whisper")
processor.save_pretrained("./saba-whisper")
!cp -r ./saba-whisper /content/drive/MyDrive/saba-whisper-finetuned/
print("Saved to Drive!")